In [2]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [3]:
!cp -r /content/gdrive/MyDrive/2390--Spring2022/2390--Spring2022/Week11/TEST/ /content/

In [4]:
!cp -r /content/gdrive/MyDrive/2390--Spring2022/2390--Spring2022/Week11/keras* /content/

In [5]:
from keras.models import load_model
from PIL import Image, ImageOps
import numpy as np

# Load the model
model = load_model('keras_model.h5')

model.summary()

Model: "sequential_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 sequential_1 (Sequential)   (None, 1280)              410208    
                                                                 
 sequential_3 (Sequential)   (None, 2)                 128300    
                                                                 
Total params: 538,508
Trainable params: 524,428
Non-trainable params: 14,080
_________________________________________________________________


In [9]:
# Create the array of the right shape to feed into the keras model
# The 'length' or number of images you can put into the array is
# determined by the first position in the shape tuple, in this case 1.
data = np.ndarray(shape=(1, 224, 224, 3), dtype=np.float32)

# Replace this with the path to your image
image = Image.open('/content/TEST/mask/TJAZTVIVPJDRFFGR7XCW6UHKQ4.jpg')

#resize the image to a 224x224 with the same strategy as in TM2:
#resizing the image to be at least 224x224 and then cropping from the center
size = (224, 224)
image = ImageOps.fit(image, size, Image.ANTIALIAS)

#turn the image into a numpy array
image_array = np.asarray(image)
# Normalize the image
normalized_image_array = (image_array.astype(np.float32) / 127.0) - 1
# Load the image into the array
data[0] = normalized_image_array

# run the inference
prediction = model.predict(data)
print(100*prediction)

[[9.997135e+01 2.865070e-02]]


# Lets try and use ImageNet based InceptionV3 as a feature extractor to see whats in the mind of the neural network that can classify 1000 different classes!

In [20]:
import tensorflow as tf
from tensorflow import keras 
from tensorflow.keras import Model
from tensorflow.keras.applications.resnet50 import ResNet50
import cv2
from tensorflow.keras.preprocessing import image
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from tensorflow.keras.applications.resnet50 import preprocess_input, decode_predictions
import matplotlib.pyplot as plt
from tensorflow.keras.layers import GlobalMaxPooling2D
import warnings
import pickle
warnings.filterwarnings('ignore')
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.inception_v3 import InceptionV3
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Flatten
from tensorflow.keras.applications import MobileNet
from tensorflow.keras.applications import DenseNet121
import matplotlib.pyplot as plt
from sklearn.feature_selection import SelectKBest
from sklearn import preprocessing
from tensorflow.keras import Sequential


def inceptionModel(height, width, print_summary=False):
    model = InceptionV3(weights='imagenet', include_top=False, input_shape = (height, width, 3))

    print(model.summary())
    model.trainable = False
    output = GlobalMaxPooling2D()(model.outputs)
    model = Model(inputs=model.inputs, outputs=output)
    if print_summary:
        model.summary()
    return model

def getFeatureVector(model, image):
    featureVector = model.predict(image)
    featureVector = featureVector.flatten()
    return featureVector

In [21]:
file_path = '/content/TEST/mask/TJAZTVIVPJDRFFGR7XCW6UHKQ4.jpg'
width = 224
height = 224
image = cv2.resize(cv2.imread(file_path), (width, height))
image = np.expand_dims(image, axis=0)

In [22]:
model = inceptionModel(width, height)
encoding = getFeatureVector(model, image)
encoding

Model: "inception_v3"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_2 (InputLayer)           [(None, 224, 224, 3  0           []                               
                                )]                                                                
                                                                                                  
 conv2d_94 (Conv2D)             (None, 111, 111, 32  864         ['input_2[0][0]']                
                                )                                                                 
                                                                                                  
 batch_normalization_94 (BatchN  (None, 111, 111, 32  96         ['conv2d_94[0][0]']              
 ormalization)                  )                                                      

array([ 0.     ,  0.     ,  0.     , ...,  0.     ,  0.     , 58.82243],
      dtype=float32)

In [16]:
encoding.shape

(10240,)

In [24]:
encoding[:120]

array([  0.       ,   0.       ,   0.       , 147.62997  ,   0.       ,
       201.8179   ,   0.       ,  62.891224 , 122.67038  , 129.25703  ,
       105.07605  ,   0.       ,   1.5255331,  88.503204 ,   0.       ,
        43.532883 , 292.7512   , 122.288994 , 201.54678  ,  61.261272 ,
        86.03803  ,   0.       ,   2.7071369,   0.       ,  51.680676 ,
       100.918076 ,  84.66843  , 113.03039  ,  86.370895 ,   0.       ,
         7.5889444,   0.       ,   0.       ,   0.       , 145.7533   ,
        31.042364 , 106.89177  ,   0.       ,   5.7183833, 167.41304  ,
       162.75954  ,  58.86389  ,  40.60874  ,  57.20328  ,   0.       ,
        63.469826 ,   0.       , 145.21028  ,  67.65688  ,  12.12845  ,
        30.469437 ,  63.637062 ,  57.974735 , 155.21898  ,  59.3899   ,
       165.31892  ,  46.418938 , 103.77747  ,   0.       ,  44.215233 ,
        63.045788 , 110.11474  , 106.31996  ,   2.5331938,   5.088363 ,
       100.61276  , 108.13272  ,  98.78902  , 149.87714  ,  75.9